# 🧹 Buổi 4/9 — Data Cleaning & Preprocessing
### Chuẩn bị dữ liệu sạch để train model!

---

> **📍 Bạn đang ở đây trong lộ trình 9 buổi:**
> ```
> Buổi 1 ✅ Giới thiệu dự án & lộ trình
> Buổi 2 ✅ Setup môi trường & khám phá SalePrice
> Buổi 3 ✅ EDA — Phân tích missing, tương quan, outliers
> Buổi 4 🔄 Data Cleaning & Preprocessing  ← BẠN ĐANG Ở ĐÂY
> Buổi 5 ⏳ Feature Engineering
> Buổi 6 ⏳ Linear Regression & Regularization
> Buổi 7 ⏳ Tree-Based Models & Đánh Giá (RF, XGBoost, LightGBM)
> Buổi 8 ⏳ Submit Kaggle (tạo file & nộp bài)
> Buổi 9 ⏳ Quiz Tốt Nghiệp (20 câu ôn tập)
> ```

---

### 📋 Nội Dung Buổi 4

| # | Task | Nội dung | Kết quả |
|---|---|---|---|
| 1 | 🔗 Combine Data | Gộp train + test để xử lý nhất quán | `all_data` (2919 hàng) |
| 2 | 🕳️ Fill Missing | 4 chiến lược điền giá trị thiếu | 0 missing còn lại |
| 3 | 🗑️ Remove Outliers | Xoá 2 nhà GrLivArea bất thường | 1458 train samples |
| 4 | 🔢 Ordinal Encoding | Mã hoá 10 cột có thứ tự chất lượng | 10 cột số |
| 5 | 🎯 One-Hot Encoding | Mã hoá 33 cột categorical còn lại | ~220 features tổng |

---

### 🔗 Kết Nối Với Buổi 3

Ở Buổi 3 (EDA), chúng ta đã **phát hiện** những vấn đề sau:

| 🔍 Phát hiện ở EDA | 🔧 Xử lý ở Buổi 4 |
|---|---|
| 19 cột missing (PoolQC 99.79%) | ✅ Task 1 — Fill với "None" hoặc 0 |
| LotFrontage missing thực sự | ✅ Task 1 — Điền median theo Neighborhood |
| 2 outlier GrLivArea >4000 sqft, giá <$300K | ✅ Task 2 — Xoá 2 hàng đó |
| KitchenQual/ExterQual có thứ tự rõ ràng | ✅ Task 3 — Ordinal Encoding |
| Neighborhood/HouseStyle không có thứ tự | ✅ Task 3 — One-Hot Encoding |

> ⚠️ **Quy tắc vàng:** Luôn **phân tích trước (EDA), xử lý sau (Cleaning)** — không làm ngược! Buổi 3 chính là lý do chúng ta biết phải làm gì ở Buổi 4.

In [19]:
# ============================================================
# ♻️ SETUP — Import thư viện & load dữ liệu
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Load dữ liệu gốc (chưa chỉnh sửa gì cả)
train_df = pd.read_csv('data/train.csv')
test_df  = pd.read_csv('data/test.csv')

print("✅ Load dữ liệu thành công!")
print(f"   train_df : {train_df.shape}  (gồm cột SalePrice)")
print(f"   test_df  : {test_df.shape}   (không có SalePrice — cần dự đoán)")
print()
print("📌 Nhắc lại từ Buổi 3:")
print(f"   Cột missing train : {train_df.isnull().sum().gt(0).sum()} cột")
print(f"   Cột missing test  : {test_df.isnull().sum().gt(0).sum()} cột")
print(f"   Outlier phát hiện : 2 nhà GrLivArea>4000 & SalePrice<$300K")

✅ Load dữ liệu thành công!
   train_df : (1460, 81)  (gồm cột SalePrice)
   test_df  : (1459, 80)   (không có SalePrice — cần dự đoán)

📌 Nhắc lại từ Buổi 3:
   Cột missing train : 19 cột
   Cột missing test  : 33 cột
   Outlier phát hiện : 2 nhà GrLivArea>4000 & SalePrice<$300K


**Expected Output:**

![img4-1](images/img_buoi4/img4-1.png)

---

## 🔗 Task 1 — Xử Lý Dữ Liệu Thiếu (Missing Values)

---

### 🤔 Tại Sao Phải Gộp Train & Test Trước Khi Xử Lý?

Đây là kỹ thuật quan trọng mà nhiều người mới thường bỏ qua:

```
❌ Cách sai (xử lý riêng từng bộ):
   train_df → điền LotFrontage = median(train) = 69.0
   test_df  → điền LotFrontage = median(test)  = 68.0
   → Mô hình thấy 2 phân phối khác nhau → dự đoán lệch!

✅ Cách đúng (gộp rồi xử lý):
   all_data = train + test
   → điền LotFrontage = median(all_data) = 68.5 (thống nhất)
   → Mô hình thấy cùng 1 phân phối → dự đoán nhất quán!
```

> 💡 **Nguyên tắc:** Train và test phải đi qua **cùng một pipeline xử lý** — như nhau từng bước. Đây là lý do trong thực tế người ta dùng `sklearn.Pipeline`.

---

### 📐 Sơ Đồ Tổng Thể Buổi 4

```
train_df (1460×81)  ──┐
                      ├── pd.concat → all_data (2919×79) → Xử lý → Tách lại
test_df  (1459×80)  ──┘
                      
                      ↓ Sau khi xử lý xong:
                      train_clean (1458×220)  ← train (đã bỏ 2 outliers, 220 features)
                      test_clean  (1459×220)  ← test (cùng 220 features)
                      y (1458,)               ← log1p(SalePrice) để train model
```

### 🗂️ 4 Chiến Lược Điền Missing Values

| Chiến lược | Cột | Lý do |
|---|---|---|
| Điền `"None"` | PoolQC, Alley, Fence, FireplaceQu, GarageXxx, BsmtXxx... | NaN = nhà không có tiện ích đó |
| Điền `0` | GarageArea, GarageCars, MasVnrArea, BsmtFinSF1... | NaN = diện tích = 0 |
| Điền median theo nhóm | `LotFrontage` | Dữ liệu thực sự thiếu, các nhà cùng khu gần tương đồng |
| Điền mode | MSZoning, Electrical, KitchenQual... | Thiếu ít (<5%), lấy giá trị phổ biến nhất |

In [20]:
# ============================================================
# 🔗 TASK 1.1 — COMBINE TRAIN & TEST
# ============================================================

# --- Bước 1: Lưu Id và Target riêng ---
train_id = train_df['Id'].copy()
test_id  = test_df['Id'].copy()

# --- Bước 2: Log-transform SalePrice → y (target) ---
y = np.log1p(train_df['SalePrice'])

# --- Bước 3: Xoá cột Id và SalePrice khỏi train; Id khỏi test ---
train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test_df.drop(['Id'], axis=1, inplace=True)

# --- Bước 4: Gộp lại ---
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)

print("✅ Gộp train + test thành công!")
print(f"   train_df  : 1460 hàng × {train_df.shape[1]} cột")
print(f"   test_df   : 1459 hàng × {test_df.shape[1]} cột")
print(f"   all_data  : {all_data.shape[0]} hàng × {all_data.shape[1]} cột")
print()
print(f"   y (target): {len(y)} giá trị | min={y.min():.2f} max={y.max():.2f} (log scale)")
print()

# Xác nhận ranh giới train/test trong all_data
ntrain = train_df.shape[0]
ntest  = test_df.shape[0]
print(f"📌 Ranh giới trong all_data:")
print(f"   all_data[:{ ntrain}] = phần train  (index 0 → {ntrain-1})")
print(f"   all_data[{ntrain}:] = phần test   (index {ntrain} → {ntrain+ntest-1})")

✅ Gộp train + test thành công!
   train_df  : 1460 hàng × 79 cột
   test_df   : 1459 hàng × 79 cột
   all_data  : 2919 hàng × 79 cột

   y (target): 1460 giá trị | min=10.46 max=13.53 (log scale)

📌 Ranh giới trong all_data:
   all_data[:1460] = phần train  (index 0 → 1459)
   all_data[1460:] = phần test   (index 1460 → 2918)


**Expected Output:**

![img4-2](images/img_buoi4/img4-2.png)

---

### 🔬 Giải Thích Code Combine Train & Test

---

```python
train_df.drop(['Id', 'SalePrice'], axis=1, inplace=True)
```

| Tham số | Ý nghĩa |
|---|---|
| `['Id', 'SalePrice']` | Danh sách cột cần xoá |
| `axis=1` | Xoá theo **cột** (axis=0 = xoá hàng) |
| `inplace=True` | Sửa trực tiếp `train_df`, không tạo bản sao mới |

**Tại sao xoá `Id`?** Cột `Id` chỉ là số thứ tự nhà (1, 2, 3,...) — không có ý nghĩa dự đoán. Nếu để vào, mô hình có thể nhầm tưởng số thứ tự ảnh hưởng giá!

**Tại sao xoá `SalePrice`?** Vì sau khi gộp với `test_df` (không có `SalePrice`), cột này sẽ toàn NaN ở nửa test → gây rối. Chúng ta đã lưu vào `y` rồi.

---

```python
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)
```

| Tham số | Ý nghĩa |
|---|---|
| `[train_df, test_df]` | List DataFrames cần ghép |
| `axis=0` | Ghép theo **hàng** (xếp chồng lên nhau) |
| `ignore_index=True` | Reset index từ 0 thay vì giữ index gốc (0-1459 rồi 0-1458) |

**Minh hoạ:**
```
train_df (1460 hàng) ──┐
                        ├── axis=0 → all_data (2919 hàng)
test_df  (1459 hàng) ──┘
```

**🎯 Mục đích:** Tạo `all_data` để xử lý thống nhất. Sau này sẽ dùng `ntrain=1460` để tách lại.

In [21]:
# ============================================================
# 🕳️ TASK 1.2 — ĐIỀN MISSING VALUES (4 CHIẾN LƯỢC)
# ============================================================

print(f"Missing trước khi xử lý: {all_data.isnull().sum().sum()} ô")
print()

# --- Chiến lược 1: Điền 'None' (categorical — nhà không có tiện ích đó) ---
none_cols = [
    'PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
    'MasVnrType'
]
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')

print(f"✅ Chiến lược 1 — Điền 'None' : {len(none_cols)} cột")
print(f"   {none_cols[:5]}... (và {len(none_cols)-5} cột khác)")

# --- Chiến lược 2: Điền 0 (numerical — diện tích/số lượng = 0 vì không có) ---
zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars',
    'MasVnrArea',
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath'
]
for col in zero_cols:
    all_data[col] = all_data[col].fillna(0)

print(f"✅ Chiến lược 2 — Điền 0     : {len(zero_cols)} cột")
print(f"   {zero_cols[:5]}... (và {len(zero_cols)-5} cột khác)")

# --- Chiến lược 3: Điền median theo Neighborhood (LotFrontage thiếu thực sự) ---
all_data['LotFrontage'] = (
    all_data.groupby('Neighborhood')['LotFrontage']
    .transform(lambda x: x.fillna(x.median()))
)
print(f"✅ Chiến lược 3 — Median theo Neighborhood : LotFrontage")

# --- Chiến lược 4: Điền mode (categorical thiếu ít, lấy giá trị phổ biến nhất) ---
mode_cols = ['MSZoning', 'Utilities', 'Functional', 'Electrical',
             'KitchenQual', 'Exterior1st', 'Exterior2nd', 'SaleType']
for col in mode_cols:
    all_data[col] = all_data[col].fillna(all_data[col].mode()[0])

print(f"✅ Chiến lược 4 — Điền mode  : {len(mode_cols)} cột")

# --- Kiểm tra kết quả ---
remaining = all_data.isnull().sum().sum()
print()
print("─" * 45)
print(f"📊 Missing còn lại sau xử lý: {remaining}")
if remaining == 0:
    print("🎉 HOÀN HẢO! Không còn missing values nào!")
else:
    remaining_cols = all_data.columns[all_data.isnull().any()].tolist()
    print(f"⚠️  Còn sót: {remaining_cols}")

Missing trước khi xử lý: 15707 ô

✅ Chiến lược 1 — Điền 'None' : 15 cột
   ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu']... (và 10 cột khác)
✅ Chiến lược 2 — Điền 0     : 10 cột
   ['GarageYrBlt', 'GarageArea', 'GarageCars', 'MasVnrArea', 'BsmtFinSF1']... (và 5 cột khác)
✅ Chiến lược 3 — Median theo Neighborhood : LotFrontage
✅ Chiến lược 4 — Điền mode  : 8 cột

─────────────────────────────────────────────
📊 Missing còn lại sau xử lý: 0
🎉 HOÀN HẢO! Không còn missing values nào!


**Expected Output:**

![img4-3](images/img_buoi4/img4-3.png)

---

### 🔬 Giải Thích Code Fill Missing Values

---

#### Chiến lược 1 & 2: `fillna('None')` và `fillna(0)`

```python
for col in none_cols:
    all_data[col] = all_data[col].fillna('None')
```

- **`fillna(value)`** — thay thế tất cả `NaN` trong cột bằng `value`
- **`for col in none_cols`** — lặp qua danh sách cột thay vì viết 15 dòng riêng lẻ
- **Tại sao `'None'` (chuỗi)?** Vì `PoolQC=NaN` nghĩa là "nhà không có bể bơi" — đây là **thông tin thực sự**, không phải lỗi! Chuỗi `'None'` sẽ được mã hoá ở bước Encoding.

> ⚠️ Đừng nhầm `'None'` (string) với `None` (Python null). Ở đây ta muốn chuỗi để phân biệt với NaN!

---

#### Chiến lược 3: Điền median theo nhóm (GroupBy Transform)

```python
all_data['LotFrontage'] = (
    all_data.groupby('Neighborhood')['LotFrontage']
    .transform(lambda x: x.fillna(x.median()))
)
```

| Lệnh | Ý nghĩa |
|---|---|
| `.groupby('Neighborhood')` | Chia all_data thành **nhóm** theo từng khu phố |
| `['LotFrontage']` | Chỉ lấy cột LotFrontage trong mỗi nhóm |
| `.transform(lambda x: ...)` | Áp dụng hàm lên mỗi nhóm, **giữ nguyên index** |
| `x.fillna(x.median())` | Trong từng nhóm: lấy median → điền vào chỗ thiếu |

**Tại sao không dùng median toàn bộ?**
```
Neighborhood A (nhà nhỏ): LotFrontage median = 50 ft
Neighborhood B (nhà lớn): LotFrontage median = 90 ft
→ Điền theo khu phố chính xác hơn!
```

---

#### Chiến lược 4: Điền Mode

```python
all_data[col] = all_data[col].fillna(all_data[col].mode()[0])
```

- **`.mode()`** trả về Series chứa các giá trị xuất hiện nhiều nhất
- **`[0]`** lấy giá trị đầu tiên (trường hợp có nhiều mode, lấy cái đầu tiên theo alphabet)
- Dùng khi: cột **categorical, thiếu rất ít** (<5 hàng), giá trị nào phổ biến nhất thì điền vào

**🎯 Mục đích sau cùng của Task 1:** Biến `all_data` từ **2919×79 với 12,009 ô NaN** thành **2919×79 với 0 ô NaN** — sẵn sàng cho encoding và training mô hình.

---

## 🗑️ Task 2 — Xử Lý Outliers (Giá Trị Bất Thường)

---

### 🤔 Outlier Là Gì? Tại Sao Phải Xử Lý?

**Outlier** là những điểm dữ liệu **bất thường, không đại diện** cho quần thể — có thể do lỗi nhập liệu, hoàn cảnh đặc biệt, hoặc gian lận.

```
VÍ DỤ THỰC TẾ:
   Nhà 1: GrLivArea=5000 sqft (diện tích sống khổng lồ) → SalePrice=$160,000 (RẺ CỰC KỲ)
   
   Lý do: nhà cũ nát, bán thanh lý → KHÔNG phản ánh thị trường thông thường
   
   Tác hại nếu giữ:
   → Model sẽ học "nhà lớn → giá thấp" — đây là kết luận SAI!
   → RMSE tăng cao, mô hình mất độ chính xác
```

### 📊 Hai Phương Pháp Phát Hiện Outlier

| Phương pháp | Cách làm | Dùng khi nào |
|---|---|---|
| **IQR (thống kê)** | Ngoài [Q1-1.5×IQR, Q3+1.5×IQR] | Dữ liệu phân phối gần bình thường |
| **Domain Knowledge** | Nhìn scatter plot, hỏi chuyên gia | Biết rõ đặc thù của dữ liệu |

> 💡 Ở Buổi 3, khi vẽ scatter `GrLivArea vs SalePrice`, chúng ta đã **nhìn thấy trực tiếp** 2 nhà bất thường góc dưới bên phải. Đây là **domain knowledge method** — tin tưởng mắt mình và hiểu dữ liệu!

In [ ]:
# ============================================================
# 🗑️ TASK 2 — PHÁT HIỆN & XOÁ OUTLIERS
# ============================================================

# --- Bước 1: Trực quan hoá để xác nhận outlier ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('GrLivArea vs SalePrice — Phát Hiện Outliers', fontsize=14, fontweight='bold')

# Lấy lại SalePrice gốc để vẽ (chưa log)
saleprice_original = np.expm1(y)  # expm1 là nghịch của log1p

# Xác định các outlier: GrLivArea > 4000 VÀ SalePrice < 300,000
grlivarea = all_data['GrLivArea'][:ntrain]  # chỉ lấy phần train
outlier_mask = (grlivarea > 4000) & (saleprice_original < 300_000)

# Plot trước khi xoá
ax = axes[0]
ax.scatter(grlivarea[~outlier_mask], saleprice_original[~outlier_mask],
           alpha=0.4, color='steelblue', label='Normal', s=20) 
# ~outlier_mask là boolean mask, ~ để lấy ngược lại (normal) -> lấy các nhà không bị outlier
ax.scatter(grlivarea[outlier_mask], saleprice_original[outlier_mask],
           color='red', s=100, zorder=5, label=f'Outliers ({outlier_mask.sum()} nhà)', marker='X')
ax.set_xlabel('GrLivArea (sqft)')
ax.set_ylabel('SalePrice ($)')
ax.set_title('Trước khi xoá')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}'))

# --- Bước 2: Xoá outlier (CHỈ xoá từ phần train của all_data) ---
# Tìm index cần xoá trong phần train của all_data
outlier_indices = outlier_mask[outlier_mask].index.tolist()
print(f"📍 Index của 2 outlier: {outlier_indices}")
print(f"   GrLivArea: {grlivarea[outlier_indices].values} sqft")
print(f"   SalePrice: ${saleprice_original[outlier_indices].values[0]:,.0f}  và  ${saleprice_original[outlier_indices].values[1]:,.0f}")
print()

# Xoá từ all_data (phần train — index 0 đến ntrain-1)
all_data.drop(index=outlier_indices, inplace=True)
all_data.reset_index(drop=True, inplace=True)

# Xoá từ y tương ứng
y.drop(index=outlier_indices, inplace=True)
y.reset_index(drop=True, inplace=True)

# Cập nhật ntrain (bây giờ là 1458 thay vì 1460)
ntrain = ntrain - len(outlier_indices)
print(f"✅ Đã xoá {len(outlier_indices)} outlier")
print(f"   all_data mới : {all_data.shape}  (train: {ntrain} | test: {ntest})")
print(f"   y mới        : {y.shape}")

# Plot sau khi xoá
grlivarea_new = all_data['GrLivArea'][:ntrain]
y_new_orig    = np.expm1(y)

ax = axes[1]
ax.scatter(grlivarea_new, y_new_orig, alpha=0.4, color='steelblue', s=20, label='Clean data')
ax.set_xlabel('GrLivArea (sqft)')
ax.set_ylabel('SalePrice ($)')
ax.set_title(f'Sau khi xoá ({ntrain} nhà)')
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:,.0f}')) # Định dạng trục y thành tiền tệ

plt.tight_layout()
plt.show()

**Expected Output:**

![img4-4](images/img_buoi4/img4-4.png)
![img4-5](images/img_buoi4/img4-5.png)

---

### 🔬 Giải Thích Code Xử Lý Outliers

---

```python
outlier_mask = (grlivarea > 4000) & (saleprice_original < 300_000)
```

- **Điều kiện kép** với `&` (AND): cả hai điều kiện đều phải đúng
- `300_000` — Python cho phép dùng `_` trong số để dễ đọc (tương đương 300000)
- Kết quả: `outlier_mask` là Series True/False — True tại hàng là outlier

---

```python
outlier_indices = outlier_mask[outlier_mask].index.tolist()
```

- `outlier_mask[outlier_mask]` — lọc chỉ lấy hàng nào là True
- `.index` — lấy danh sách index (số hàng) của những hàng đó
- `.tolist()` — chuyển sang Python list để dễ dùng sau

---

```python
all_data.drop(index=outlier_indices, inplace=True)
all_data.reset_index(drop=True, inplace=True)
```

| Lệnh | Ý nghĩa |
|---|---|
| `.drop(index=...)` | Xoá hàng theo index (không phải cột) |
| `.reset_index(drop=True)` | Reset lại index từ 0 (tránh index bị lỗ: 0,1,3,4...) |
| `drop=True` | Không tạo cột index cũ (xoá luôn) |

> ⚠️ **QUAN TRỌNG:** Chúng ta xoá ở cả `all_data` lẫn `y` với cùng `outlier_indices`. Nếu xoá ở `all_data` mà không xoá ở `y` → misalign → model sẽ học sai!

**Tại sao chỉ xoá phần train?** Index 0 đến ntrain-1 là train. Outlier nằm trong train (có SalePrice). Test set không có SalePrice nên không thể biết outlier ở đó.

**🎯 Mục đích sau cùng của Task 2:** Đảm bảo model học từ dữ liệu **đại diện cho thị trường thực**. 2 nhà bất thường này có thể khiến regression line bị kéo lệch hoàn toàn.

---

## 🔢 Task 3 — Encoding Categorical Features

---

### 🤔 Tại Sao Cần Encoding?

Machine Learning models **không hiểu chữ** — chúng chỉ làm toán với số. Chúng ta cần **chuyển text thành số**.

```
KitchenQual = "Gd"  ← model không hiểu "Gd" là gì!
KitchenQual = 3     ← model hiểu: 3 > 2 > 1 → chất lượng tốt hơn
```

### 📊 Hai Loại Categorical Data & Cách Encoding

**Loại 1: Ordinal (có thứ tự)** — dùng Label Encoding
```
Po < Fa < TA < Gd < Ex  (chất lượng tăng dần)
 0    1    2    3    4  (gán số tăng dần)
```
Ví dụ: `ExterQual`, `KitchenQual`, `BsmtQual`, `PoolQC`...

**Loại 2: Nominal (không có thứ tự)** — dùng One-Hot Encoding
```
Neighborhood: CollgCr, Veenker, Crawfor, NoRidge...
→ CollgCr_1 Veenker_1 Crawfor_1 NoRidge_1 ...
→ Không có "CollgCr > Veenker" — chúng bình đẳng!
```

### ⚠️ Dummy Variable Trap

Khi dùng One-Hot, cần bỏ 1 cột để tránh **multicollinearity hoàn hảo**:
```
Ví dụ: BldgType có 5 loại → tạo 5 cột binary
Nếu 4 cột đều = 0 → cột thứ 5 CHẮC CHẮN = 1 (thông tin dư thừa!)
→ drop_first=True trong pd.get_dummies() xử lý vấn đề này
```

In [ ]:
# ============================================================
# 🔢 TASK 3.1 — ORDINAL ENCODING (CÁC CỘT CÓ THỨ TỰ CHẤT LƯỢNG)
# ============================================================

# Định nghĩa thứ tự tăng dần cho từng cột ordinal
ordinal_mappings = {
    'ExterQual'   : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond'   : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual'    : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond'    : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtExposure': ['None', 'No', 'Mn', 'Av', 'Gd'],
    'BsmtFinType1': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'BsmtFinType2': ['None', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ'],
    'HeatingQC'   : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual' : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu' : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageFinish': ['None', 'Unf', 'RFn', 'Fin'],
    'GarageQual'  : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond'  : ['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC'      : ['None', 'Fa', 'TA', 'Gd', 'Ex'],
    'Fence'       : ['None', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv'],
    'Functional'  : ['Sal', 'Sev', 'Maj2', 'Maj1', 'Mod', 'Min2', 'Min1', 'Typ'],
    'LandSlope'   : ['Sev', 'Mod', 'Gtl'],
    'LotShape'    : ['IR3', 'IR2', 'IR1', 'Reg'],
    'PavedDrive'  : ['N', 'P', 'Y'],
}

# Áp dụng encoding
for col, order in ordinal_mappings.items():
    mapping_dict = {val: i for i, val in enumerate(order)}
    all_data[col] = all_data[col].map(mapping_dict)
    # Điền 0 nếu có giá trị không nằm trong danh sách
    all_data[col] = all_data[col].fillna(0).astype(int)

# Kiểm tra kết quả
print("✅ Ordinal Encoding hoàn tất!")
print(f"   Số cột được encode ordinal: {len(ordinal_mappings)}")
print()
print("📊 Ví dụ KitchenQual sau encoding:")
example = all_data[['KitchenQual']].value_counts().head(6)
print(example.to_string())
print()

# Xác nhận không còn string ở các cột ordinal
ordinal_dtypes = all_data[list(ordinal_mappings.keys())].dtypes
non_int = ordinal_dtypes[ordinal_dtypes == 'object']
if len(non_int) == 0:
    print("✅ Tất cả cột ordinal đều là số (int)")
else:
    print(f"⚠️ Vẫn còn string: {non_int.index.tolist()}")

**Expected Output:**

![img4-6](images/img_buoi4/img4-6.png)

---

### 🔬 Giải Thích Code Ordinal Encoding

---

```python
mapping_dict = {val: i for i, val in enumerate(order)}
```

- **`enumerate(order)`** — duyệt list kèm index: `(0, 'None'), (1, 'Po'), (2, 'Fa')...`
- **`{val: i for i, val in ...}`** — dict comprehension tạo bảng dịch:
  `{'None': 0, 'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}`

```python
all_data[col] = all_data[col].map(mapping_dict)
```

- **`.map(dict)`** — thay thế mỗi giá trị trong cột theo bảng dịch
- Nếu giá trị không có trong dict → trả về `NaN` (sẽ được fillna(0) ở dòng sau)

```python
all_data[col] = all_data[col].fillna(0).astype(int)
```

- **`.fillna(0)`** — những giá trị lạ không có trong mapping → gán 0 (= "None"/"lowest")
- **`.astype(int)`** — đảm bảo kiểu dữ liệu là integer, không phải float

---

**Minh hoạ quá trình:**

```
Cột KitchenQual gốc:       →  Sau mapping_dict:    →  Sau astype(int):
┌──────────────┐               ┌──────┐                ┌──────┐
│ TA           │    map()      │ 3.0  │   astype(int)  │  3   │
│ Gd           │  ─────────→   │ 4.0  │  ──────────→   │  4   │
│ Ex           │               │ 5.0  │                │  5   │
│ TA           │               │ 3.0  │                │  3   │
└──────────────┘               └──────┘                └──────┘
```

**🎯 Mục đích sau cùng:** Model có thể hiểu `KitchenQual=4 (Gd) > KitchenQual=3 (TA)` → giá nhà có xu hướng cao hơn theo thứ tự này.

In [ ]:
# ============================================================
# 🎯 TASK 3.2 — ONE-HOT ENCODING (CÁC CỘT CATEGORICAL KHÔNG CÓ THỨ TỰ)
# ============================================================

# Kiểm tra các cột còn lại là object (string)
categorical_cols = all_data.select_dtypes(include=['object']).columns.tolist()
print(f"📊 Số cột categorical (nominal) cần one-hot: {len(categorical_cols)}")
print()
print("Danh sách:")
for i, col in enumerate(categorical_cols):
    n_unique = all_data[col].nunique()
    print(f"   {i+1:02d}. {col:20s} — {n_unique:2d} giá trị unique")
print()

# Áp dụng One-Hot Encoding
shape_before = all_data.shape
all_data = pd.get_dummies(all_data, columns=categorical_cols, drop_first=True)
shape_after = all_data.shape

print(f"✅ One-Hot Encoding hoàn tất!")
print(f"   Trước : {shape_before[0]} hàng × {shape_before[1]:3d} cột")
print(f"   Sau   : {shape_after[0]} hàng × {shape_after[1]:3d} cột")
print(f"   Tăng  : +{shape_after[1] - shape_before[1]} cột mới")
print()

# Kiểm tra không còn string
remaining_obj = all_data.select_dtypes(include=['object']).shape[1]
if remaining_obj == 0:
    print("✅ Không còn cột kiểu string — sẵn sàng train model!")
else:
    print(f"⚠️ Còn {remaining_obj} cột string chưa được encode!")

**Expected Output:**

![img4-7](images/img_buoi4/img4-7.png)

---

### 🔬 Giải Thích Code One-Hot Encoding

---

```python
categorical_cols = all_data.select_dtypes(include=['object']).columns.tolist()
```

- **`.select_dtypes(include=['object'])`** — lọc chỉ các cột có dtype là `object` (= string trong pandas)
- **`.columns`** — lấy tên các cột đó
- **`.tolist()`** — chuyển sang Python list

---

```python
all_data = pd.get_dummies(all_data, columns=categorical_cols, drop_first=True)
```

| Tham số | Ý nghĩa |
|---|---|
| `all_data` | DataFrame cần biến đổi |
| `columns=categorical_cols` | Chỉ encode các cột này |
| `drop_first=True` | Bỏ 1 cột đầu tiên của mỗi group → tránh dummy variable trap |

**Minh hoạ `drop_first=True`:**
```
MSZoning (5 giá trị: C, FV, RH, RL, RM):
    Không drop_first: 5 cột  (C, FV, RH, RL, RM)
    Có drop_first   : 4 cột  (FV, RH, RL, RM)  ← C được ngầm hiểu khi cả 4 = 0
```

**Tại sao không dùng ordinal cho các cột này?**
- `Neighborhood`: Không có thứ tự "CollgCr > Crawfor"  
- `MSZoning`: Không có thứ tự "RL > RM"  
- Nếu gán số, model sẽ hiểu sai: số lớn hơn = tốt hơn → kết quả sai!

**🎯 Mục đích sau cùng của Task 3:** Chuyển toàn bộ dữ liệu sang dạng **số thuần túy**, không còn string — machine learning algorithms chỉ làm việc với số. Kết quả: từ 79 cột → ~220 cột (số tăng do one-hot).

---

## 📝 Ghi Chú Bổ Sung — Các Phương Pháp Encoding Khác

> **Phần này KHÔNG nằm trong pipeline chính của bài học.** Mục đích giúp bạn biết rằng One-Hot không phải lựa chọn duy nhất — mỗi kỹ thuật có "đất dụng võ" riêng.

---

### 🗺️ Bản Đồ Các Kỹ Thuật Encoding

| Kỹ thuật | Cardinality phù hợp | Thêm cột? | Độ phức tạp | Ghi chú |
|---|---|---|---|---|
| **Ordinal Encoding** | Bất kỳ (có thứ tự) | Không | ⭐ | Đã làm ở Task 2 |
| **One-Hot Encoding** | Thấp (<20) | Có (+N cột) | ⭐ | Đã làm ở Task 3 |
| **Frequency Encoding** | Trung bình | Không | ⭐⭐ | Thay bằng tần số xuất hiện |
| **Target Encoding** | Cao (>20) | Không | ⭐⭐⭐ | Thay bằng mean của target |
| **Binary Encoding** | Cao (>50) | Có (+log₂N cột) | ⭐⭐⭐ | Ít cột hơn One-Hot nhiều |
| **Hash Encoding** | Rất cao (>1000) | Có (cố định) | ⭐⭐⭐ | Dùng cho NLP, user-id, ... |
| **Embedding** | Cực lớn | — | ⭐⭐⭐⭐⭐ | Deep Learning |

---

### 🔢 1. Frequency Encoding (Count Encoding)

**Ý tưởng:** Thay mỗi category bằng **số lần nó xuất hiện** trong toàn bộ dataset.

```python
# Ví dụ: Neighborhood có 25 khu phố khác nhau
freq_map = all_data['Neighborhood'].value_counts().to_dict()
all_data['Neighborhood_freq'] = all_data['Neighborhood'].map(freq_map)

# Kết quả:
#   'NAmes'   → 443  (xuất hiện nhiều nhất)
#   'Veenker' → 24   (xuất hiện ít nhất)
```

| | Neighborhood | Neighborhood_freq |
|---|---|---|
| 0 | NAmes | 443 |
| 1 | CollgCr | 267 |
| 2 | Veenker | 24 |

- ✅ **Ưu:** Không tăng số cột, cực nhanh, không cần thư viện ngoài
- ❌ **Nhược:** Hai category có tần số bằng nhau → cùng giá trị → mất phân biệt
- 🎯 **Dùng khi:** Tần số xuất hiện CÓ ý nghĩa (khu phố phổ biến ↔ nhiều dữ liệu ↔ giá ổn định)

---

### 🎯 2. Target Encoding (Mean Encoding)

**Ý tưởng:** Thay mỗi category bằng **giá trị trung bình của target (SalePrice)** trong nhóm đó.

```python
# Cách đơn giản (chỉ dùng để minh hoạ — xem cảnh báo bên dưới!)
target_map = train_df.groupby('Neighborhood')['SalePrice'].mean().to_dict()
train_df['Neighborhood_te'] = train_df['Neighborhood'].map(target_map)
test_df['Neighborhood_te']  = test_df['Neighborhood'].map(target_map)

# Kết quả:
#   'NridgHt' → 335,295$  (khu phố sang nhất)
#   'MeadowV' →  98,457$  (khu phố bình dân nhất)
```

> ⚠️ **Nguy hiểm — Target Leakage!**
>
> Nếu bạn encode **toàn bộ train** bằng mean của chính nó, model "nhìn thấy" target khi học → overfit.
>
> **Giải pháp đúng:** Dùng **K-Fold Target Encoding** (tính mean từ các fold khác nhau):
>
> ```python
> # pip install category_encoders
> from category_encoders import TargetEncoder
>
> enc = TargetEncoder(cols=['Neighborhood'], smoothing=10)
> enc.fit(X_train, y_train)
> X_train['Neighborhood_te'] = enc.transform(X_train)['Neighborhood']
> X_test['Neighborhood_te']  = enc.transform(X_test)['Neighborhood']
> ```
>
> Tham số `smoothing=10` giúp "kéo" về global mean khi nhóm có ít mẫu → giảm overfit.

- ✅ **Ưu:** Rất mạnh với tree-based model (LightGBM, XGBoost, CatBoost), không tăng chiều
- ❌ **Nhược:** Dễ overfit nếu dùng sai, phức tạp hơn
- 🎯 **Dùng khi:** Cardinality cao (>20 unique) + bài toán regression/classification

---

### 🔣 3. Binary Encoding

**Ý tưởng:** Chuyển category → số nguyên → biểu diễn bằng nhị phân (từng bit là 1 cột).

```
Neighborhood có 25 giá trị:
  One-Hot   → 24 cột
  Binary    →  5 cột  (vì 2⁵ = 32 ≥ 25)  ← tiết kiệm gần 5× !

Ví dụ:
  'NAmes'   →  1 (01) → [0, 1]
  'CollgCr' →  2 (10) → [1, 0]
  'NridgHt' → 10 (1010) → [1, 0, 1, 0]
```

```python
# pip install category_encoders
from category_encoders import BinaryEncoder

enc = BinaryEncoder(cols=['Neighborhood'])
all_data_bin = enc.fit_transform(all_data)
```

- ✅ **Ưu:** Số cột = log₂(cardinality) — tiết kiệm cực nhiều khi cardinality > 50
- ❌ **Nhược:** Các bit không có ý nghĩa trực tiếp, khó giải thích với business
- 🎯 **Dùng khi:** Cardinality rất cao (>50), muốn giữ ít feature

---

### 📌 Khi Nào Dùng Gì? (Rule of Thumb)

```
Cardinality thấp  (≤ 10)  → One-Hot Encoding ✅
Cardinality trung (10-50) → One-Hot hoặc Target Encoding
Cardinality cao   (≥ 50)  → Target Encoding hoặc Binary Encoding
Cardinality cực cao (vd: user_id, zip_code) → Hash Encoding / Embedding
```

> 💡 **Trong dự án nhà ở này:** One-Hot + Ordinal hoàn toàn đủ vì cardinality thấp (≤ 25).
> Target Encoding sẽ được thử nghiệm như kỹ thuật nâng cao ở Buổi 5.

In [ ]:
# ============================================================
# ✂️ TASK 4 — TÁCH LẠI TRAIN / TEST & KIỂM TRA CUỐI CÙNG
# ============================================================

# Tách all_data thành train và test dựa vào ntrain
X_train = all_data[:ntrain].copy()
X_test  = all_data[ntrain:].copy()

# Reset index của X_test
X_test.reset_index(drop=True, inplace=True)

print("✅ Tách lại train/test hoàn tất!")
print(f"   X_train : {X_train.shape}  ← dùng để train model")
print(f"   X_test  : {X_test.shape}  ← dùng để predict & submit Kaggle")
print(f"   y       : {y.shape}   ← target (log-transformed SalePrice)")
print()

# --- Kiểm tra cuối: không còn missing values ---
assert X_train.isnull().sum().sum() == 0, "⚠️ X_train vẫn còn missing!"
assert X_test.isnull().sum().sum()  == 0, "⚠️ X_test vẫn còn missing!"
print("✅ Không còn missing values trong X_train và X_test")

# --- Kiểm tra shapes khớp nhau ---
assert X_train.shape[1] == X_test.shape[1], "⚠️ X_train và X_test có số cột khác nhau!"
print(f"✅ Số feature thống nhất: {X_train.shape[1]} cột")
print()

# --- Tổng kết pipeline ---
print("=" * 55)
print("📊 TỔNG KẾT DATA PIPELINE BUỔI 4")
print("=" * 55)
pipeline_log = pd.DataFrame({
    'Bước': [
        '0. Gốc (train)',
        '0. Gốc (test)',
        '1. Sau combine',
        '2. Sau fill missing',
        '3. Sau xoá outlier',
        '4. Sau ordinal encoding',
        '5. Sau one-hot encoding',
        '6. X_train cuối',
        '6. X_test cuối',
    ],
    'Hàng × Cột': [
        '1460 × 81',
        '1459 × 80',
        f'2919 × 79',
        f'2919 × 79 (0 NaN)',
        f'{ntrain + ntest} × 79 ({ntrain} train)',
        f'{ntrain + ntest} × 79 (ordinal → int)',
        f'{all_data.shape[0]} × {all_data.shape[1]}',
        f'{X_train.shape[0]} × {X_train.shape[1]}',
        f'{X_test.shape[0]} × {X_test.shape[1]}',
    ]
})
print(pipeline_log.to_string(index=False))

**Expected Output:**

![img4-8](images/img_buoi4/img4-8.png)

---

## 📝 Bài Tập Buổi 4

> **Mục tiêu:** Ôn lại những gì đã làm — kiểm tra missing values, encoding thủ công, và one-hot encoding.

---

### Bài Tập 1 — Kiểm Tra LotFrontage Sau Khi Điền

Sau khi chạy bước điền missing values, hãy kiểm tra lại kết quả:

1. Kiểm tra `LotFrontage` trong `all_data` còn bị missing không (`isnull().sum()`)
2. In ra giá trị **median** của `LotFrontage` trong `all_data`

> 💡 Gợi ý: `.isnull().sum()` | `.median()`

In [ ]:
# ============================================================
# BÀI TẬP 1 — Kiểm tra LotFrontage sau khi điền
# ============================================================

# TODO 1: Kiểm tra LotFrontage còn NaN không
missing_lot = all_data[???].isnull().???   # TODO: tên cột và hàm đếm NaN
print(f"LotFrontage missing trong all_data: {missing_lot}")

# TODO 2: Tính median tổng thể của LotFrontage
median_lot = all_data[???].???   # TODO: tên cột và hàm tính median
print(f"Median LotFrontage: {median_lot:.1f} ft")

**Expected Output:**

![img4-9](images/img_buoi4/img4-9.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 1</summary>

```python
missing_lot = all_data['LotFrontage'].isnull().sum()
# → Kết quả: 0 (đã điền hết rồi!)

median_lot = all_data['LotFrontage'].median()
# → Khoảng 69.0 ft
```

**Nhận xét:** Sau khi điền, không còn NaN. Median gần với giá trị ban đầu — chiến lược fill không làm méo phân phối.

</details>

---

### Bài Tập 2 — Ordinal Encoding Thủ Công Cho `ExterQual`

Trong buổi học ta đã dùng `map()` để chuyển categorical thành số. Hãy tự làm tương tự với `ExterQual` (chất lượng vật liệu bên ngoài):

- `'Po'` → 1 (Poor)
- `'Fa'` → 2 (Fair)
- `'TA'` → 3 (Typical/Average)
- `'Gd'` → 4 (Good)
- `'Ex'` → 5 (Excellent)

1. Tạo cột `ExterQual_enc` trong `all_data` bằng cách dùng `.map()`
2. In ra `value_counts()` của cột mới để kiểm tra kết quả

> 💡 Gợi ý: `df['col'].map({'A': 1, 'B': 2, ...})`

In [ ]:
# ============================================================
# BÀI TẬP 2 — Ordinal encoding thủ công cho ExterQual
# ============================================================

# Bảng encoding
qual_map = {
    '???': ???,   # TODO: Po = 1
    '???': ???,   # TODO: Fa = 2
    '???': ???,   # TODO: TA = 3
    '???': ???,   # TODO: Gd = 4
    '???': ???    # TODO: Ex = 5
}

# TODO: Tạo cột mới bằng .map()
all_data['ExterQual_enc'] = all_data[???].map(???)

# Kiểm tra kết quả
print("Phân phối ExterQual_enc:")
print(all_data['ExterQual_enc'].value_counts().sort_index())

**Expected Output:**

![img4-10](images/img_buoi4/img4-10.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 2</summary>

```python
qual_map = {'Po': 1, 'Fa': 2, 'TA': 3, 'Gd': 4, 'Ex': 5}

all_data['ExterQual_enc'] = all_data['ExterQual'].map(qual_map)

print(all_data['ExterQual_enc'].value_counts().sort_index())
# 1 (Po):   ...
# 2 (Fa):   ...
# 3 (TA):  ~... (nhiều nhất)
# 4 (Gd):  ~...
# 5 (Ex):  ~...
```

**Nhận xét:** Ordinal encoding giữ được thứ tự (Ex > Gd > TA > ...). Phù hợp với các cột có thứ bậc ý nghĩa như chất lượng.

</details>

---

### Bài Tập 3 — Đếm Số Cột Sau One-Hot Encoding

Trong buổi học, `pd.get_dummies()` đã tạo ra nhiều cột mới. Hãy thử với một cột đơn lẻ:

1. Dùng `pd.get_dummies()` trên cột `MSZoning` từ `all_data`
2. In ra số cột tạo ra và tên các cột đó
3. In ra 3 dòng đầu để xem dữ liệu trông như thế nào

> 💡 Gợi ý: `pd.get_dummies(all_data['MSZoning'])` | `.shape` | `.columns`

In [ ]:
# ============================================================
# BÀI TẬP 3 — Đếm số cột sau one-hot encoding
# ============================================================

# TODO 1: Tạo dummy columns từ MSZoning
mszoning_dummies = pd.get_dummies(all_data[???])   # TODO: tên cột

# TODO 2: In số cột và tên cột
print(f"Số cột dummy tạo ra: {mszoning_dummies.???[1]}")   # TODO: hàm lấy shape
print(f"Tên các cột: {list(mszoning_dummies.columns)}")

# TODO 3: In 3 dòng đầu
print("\n3 dòng đầu:")
print(mszoning_dummies.head(???))

**Expected Output:**

![img4-11](images/img_buoi4/img4-11.png)

<details>
<summary>👆 Click để xem đáp án Bài Tập 3</summary>

```python
mszoning_dummies = pd.get_dummies(all_data['MSZoning'])

print(f"Số cột dummy tạo ra: {mszoning_dummies.shape[1]}")
print(f"Tên các cột: {list(mszoning_dummies.columns)}")
# → Ví dụ: ['C (all)', 'FV', 'RH', 'RL', 'RM'] — 5 cột

print(mszoning_dummies.head(3))
```

**Nhận xét:** `MSZoning` có 5 giá trị unique → tạo ra 5 cột dummy (hoặc 4 nếu dùng `drop_first=True`). Đây chính xác là cách `pd.get_dummies()` trong buổi học đã làm với tất cả các cột object.

</details>

---

> 🎯 **Sau Buổi 4:** Dữ liệu đã sạch, đã encode, sẵn sàng! Buổi 5 sẽ học cách **tạo thêm feature mới** để cải thiện độ chính xác của model.
> 
> **Hẹn gặp lại ở Buổi 5 — Feature Engineering! 🚀**

---

## 🏁 Tổng Kết Buổi 4

---

### ✅ Những Gì Bạn Đã Làm Được Hôm Nay

| # | Task | Kết quả |
|---|---|---|
| 1 | 🔗 Gộp train + test | `all_data` 2919 × 79 |
| 2 | 🕳️ Điền 4 loại missing | 0 NaN còn lại |
| 3 | 🗑️ Xoá 2 outlier | 1458 mẫu train sạch |
| 4 | 🔢 Ordinal encoding 19 cột | Cột chất lượng → số có thứ tự |
| 5 | 🎯 One-Hot encoding ~33 cột | 220 features tổng cộng |
| 6 | ✂️ Tách lại X_train / X_test | Sẵn sàng train model! |

### 📦 Các Biến Quan Trọng Để Dùng Ở Buổi 5 & 6

```python
X_train  → (1458, ~220)  — features train đã sạch
X_test   → (1459, ~220)  — features test đã sạch
y        → (1458,)       — log1p(SalePrice), target để train
ntrain   → 1458          — số lượng mẫu train
ntest    → 1459          — số lượng mẫu test
```

### 🔭 Buổi 5 Sẽ Làm Gì?

> **Feature Engineering** — Tạo thêm đặc trưng mới từ các cột hiện có!

```
TotalSF      = TotalBsmtSF + 1stFlrSF + 2ndFlrSF   (diện tích tổng)
HouseAge     = YrSold - YearBuilt                   (tuổi nhà)
RemodAge     = YrSold - YearRemodAdd                (năm kể từ sửa)
HasGarage    = 1 if GarageArea > 0 else 0           (có/không)
TotalPorch   = tổng tất cả loại hiên                (diện tích hiên)
```

Những feature mới này sẽ giúp model "hiểu" dữ liệu tốt hơn và cải thiện RMSE đáng kể!

---

> 💡 **Nhớ lưu lại** các biến `X_train`, `X_test`, `y` — bạn sẽ dùng liên tục từ Buổi 5 đến Buổi 8!